# 02 – Data preparation: Secondary Mushroom

**Worked on by:** Andreas Schellekens (cleaning steps, preprocessing pipeline, sanity checks)

> **GenAI disclosure:** the first version of this notebook was drafted with the help of Claude Code (AI assistant), based on the decisions we made in `01_eda.ipynb`. All steps were reviewed by the team; we must be able to explain every cell ourselves.

## 1. Setup & loading

Imports, file paths and constants. `RANDOM_STATE` is fixed so the train/test split is the same every time the notebook runs (and the same for every teammate).
`LOKY_MAX_CPU_COUNT` silences a harmless joblib warning on Windows (see `CLAUDE.md`); it must be set before scikit-learn is imported.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

DATA_DIR = Path("Data")
MODEL_DIR = Path("models")
RAW_PATH = DATA_DIR / "mushroom_project_dataset.csv"
CLEANED_PATH = DATA_DIR / "mushroom_cleaned.csv"
PREPARED_TRAIN_PATH = DATA_DIR / "mushroom_prepared_train.csv"
PREPARED_TEST_PATH = DATA_DIR / "mushroom_prepared_test.csv"
PREPROCESSOR_PATH = MODEL_DIR / "mushroom_preprocessor.joblib"

RANDOM_STATE = 42
TEST_SIZE = 0.20

We read the raw CSV. It is only read, never overwritten, so the original stays available.
The `assert` checks that the file is the one we explored in the EDA (5000 rows, 13 columns). If the lecturer ever hands out a new version, the notebook stops here instead of silently producing something different.

In [2]:
df_raw = pd.read_csv(RAW_PATH)
assert df_raw.shape == (5000, 13), f"Unexpected shape {df_raw.shape} – is this the dataset from the EDA?"
print(f"Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

Raw data: 5,000 rows × 13 columns


## 2. Column names & readable labels
*(EDA section 2 – decision table steps 1 and 2)*

- Column names: `cap-diameter` → `cap_diameter` (snake_case works as a Python attribute and as a JSON key in the API).
- The categorical values are one-letter codes. We translate them with the UCI dictionary, so later outputs (feature importances, the web interface) are readable.
- The code `f` means **"none"** for `gill_color`, `stem_surface` and `ring_type` (no gills / no stem surface / no ring). That is real information, not a missing value, so it becomes the label `none` – different from `NaN`.

The noise columns are not translated; we drop them in the next step anyway.

In [3]:
COLOR = {"n": "brown", "b": "buff", "g": "gray", "r": "green", "p": "pink", "u": "purple",
         "e": "red", "w": "white", "y": "yellow", "l": "blue", "o": "orange", "k": "black", "f": "none"}

CODE_MAPS = {
    "class":             {"e": "edible", "p": "poisonous"},
    "cap_shape":         {"b": "bell", "c": "conical", "x": "convex", "f": "flat",
                          "s": "sunken", "p": "spherical", "o": "others"},
    "gill_color":        COLOR,
    "spore_print_color": COLOR,
    "stem_surface":      {"i": "fibrous", "g": "grooves", "y": "scaly", "s": "smooth", "h": "shiny",
                          "l": "leathery", "k": "silky", "t": "sticky", "w": "wrinkled", "e": "fleshy", "f": "none"},
    "ring_type":         {"c": "cobwebby", "e": "evanescent", "r": "flaring", "g": "grooved", "l": "large",
                          "p": "pendant", "s": "sheathing", "z": "zone", "y": "scaly", "m": "movable", "f": "none"},
    "habitat":           {"g": "grasses", "l": "leaves", "m": "meadows", "p": "paths", "h": "heaths",
                          "u": "urban", "w": "waste", "d": "woods"},
    "season":            {"s": "spring", "u": "summer", "a": "autumn", "w": "winter"},
}

We work on a copy (`df`) so `df_raw` stays untouched for comparison.
Before translating, we check that every code in the data exists in its dictionary: `.map()` turns an unknown code into `NaN` without any warning, which would silently create extra missing values.

In [4]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")

for col, mapping in CODE_MAPS.items():
    unknown = set(df[col].dropna().unique()) - set(mapping)
    assert not unknown, f"{col} has unmapped codes: {unknown}"
    df[col] = df[col].map(mapping)

# Translating must not create or remove missing values
assert df.isna().sum().sum() == df_raw.isna().sum().sum()
df.columns.tolist()

['cap_diameter',
 'stem_height',
 'stem_width',
 'spore_print_color',
 'gill_color',
 'habitat',
 'season',
 'ring_type',
 'cap_shape',
 'stem_surface',
 'jumbled_noise_0',
 'jumbled_noise_1',
 'class']

## 3. Drop the noise columns
*(EDA section 4 – step 3)*

`jumbled_noise_0` and `jumbled_noise_1` are row-shuffled copies of `cap_shape`: identical value counts, agreement with `cap_shape` at chance level, and no association with the class after a Bonferroni correction. Keeping them only gives a model the opportunity to fit noise. Section 9 below checks that dropping them indeed helps.

In [5]:
NOISE_COLS = ["jumbled_noise_0", "jumbled_noise_1"]
df = df.drop(columns=NOISE_COLS)

NUM_COLS = ["cap_diameter", "stem_height", "stem_width"]
CAT_COLS = ["spore_print_color", "gill_color", "habitat", "season", "ring_type", "cap_shape", "stem_surface"]
TARGET = "class"
assert set(df.columns) == set(NUM_COLS + CAT_COLS + [TARGET])

## 4. Missing values in categoricals
*(EDA section 5 – steps 4 and 5)*

- We **keep every row**. Only 12 of the 5000 rows are complete, so `dropna()` would throw away almost the whole dataset.
- For categoricals, `NaN` becomes the explicit category `"missing"`. For `stem_surface` and `spore_print_color` the fact that the value is missing is related to the class (EDA 5.2), so the model must be able to *see* it. For the other categoricals it is harmless: the model can simply learn that `"missing"` means nothing special.
- `spore_print_color` is 95% missing. Imputing it would mean inventing almost the whole column, so `"missing"` is the only honest option. Whether to drop the column entirely is an open question for the model notebooks (it is easy to drop there).
- Numerical `NaN` are **not** filled here: the median must be learned on the training set only (section 7).

In [6]:
df[CAT_COLS] = df[CAT_COLS].fillna("missing")
df[CAT_COLS].isna().sum().sum(), df[NUM_COLS].isna().sum().to_dict()

(0, {'cap_diameter': 2032, 'stem_height': 1013, 'stem_width': 395})

## 5. Stem consistency & the `has_stem` feature
*(EDA section 6.1 – step 7)*

In the EDA we found that `stem_height = 0` and `stem_width = 0` always occur together: these are mushrooms **without a stem**, and 92% of them are poisonous. Three extra facts, checked while building this notebook:
- `stem_surface = none` occurs **only** on stemless mushrooms (56 of 56 rows) – it is a third way to recognise them.
- Some stemless mushrooms miss one of the two measures (11 rows have width 0 but height missing, 3 the other way round).
- For 83 rows both measures are missing, but for some of those `stem_surface` still tells us whether there is a stem.

Why this matters: median imputation (section 7) would give a stemless mushroom a height of ~6 cm, i.e. turn it into a "normal" mushroom and erase the strongest signal in the numericals. So before imputing, we use the information we *do* have:

1. `has_stem = 0` if any of the three sources says "no stem"; `has_stem = 1` if any says there is a stem (a measure > 0 or a real stem surface); otherwise unknown (`NaN`, imputed in section 7).
2. For stemless mushrooms, set a missing stem measure to 0 and a missing `stem_surface` to `none`. These are logical consequences, not guesses.

First we check that the three sources never contradict each other; if they did, one of them would be wrong and we couldn't trust this rule.

In [7]:
height, width, surface = df["stem_height"], df["stem_width"], df["stem_surface"]
says_no_stem = (height == 0) | (width == 0) | (surface == "none")
says_stem = (height > 0) | (width > 0) | ~surface.isin(["none", "missing"])

assert not (says_no_stem & says_stem).any(), "Contradicting stem information"
print(f"No stem: {says_no_stem.sum()} | stem: {says_stem.sum()} | unknown: {(~says_no_stem & ~says_stem).sum()}")

No stem: 60 | stem: 4887 | unknown: 53


Now we apply the rule. `has_stem` uses pandas' nullable integer type `Int64`, so it can be 0, 1 or missing (written as an empty cell in the CSV).
We print how many values the logical fill repaired.

In [8]:
filled = {
    "stem_height": (says_no_stem & height.isna()).sum(),
    "stem_width": (says_no_stem & width.isna()).sum(),
    "stem_surface": (says_no_stem & (surface == "missing")).sum(),
}
df.loc[says_no_stem, ["stem_height", "stem_width"]] = 0.0
df.loc[says_no_stem, "stem_surface"] = "none"
df["has_stem"] = pd.Series(pd.NA, index=df.index, dtype="Int64")  # start as unknown
df.loc[says_stem, "has_stem"] = 1
df.loc[says_no_stem, "has_stem"] = 0

print("Values filled by the stem rule:", filled)
print(pd.crosstab(df["has_stem"].astype("object").fillna("unknown"), df[TARGET], margins=True))

Values filled by the stem rule: {'stem_height': 11, 'stem_width': 3, 'stem_surface': 4}
class     edible  poisonous   All
has_stem                         
0              4         56    60
1           3070       1817  4887
unknown       33         20    53
All         3107       1893  5000


**Outliers** *(EDA 6.3 – step 7)*: we deliberately do **nothing**. The large values are biologically plausible (caps of 30–60 cm exist), removing them would cost 5–10% of an already small dataset, and there are no impossible values. We only verify that last claim, so the notebook fails if a future version of the data contains negative sizes.

In [9]:
assert (df[NUM_COLS].dropna() >= 0).all().all(), "Negative sizes found – physically impossible"

**Ordered categoricals** *(EDA 7.3)*: none of the categoricals has a natural order (`season` is cyclic), so they stay nominal and will be one-hot encoded – no ordinal encoding.

## 6. Train/test split & saving the cleaned dataset
*(EDA section 3 – step 10)*

We split **once, here**, and store which row belongs to which set. Every model notebook (baseline, PyCaret, tuned models, SageMaker) then uses the **same** test set, so their scores can be compared fairly in `07_model_comparison.ipynb`.

- 80% train / 20% test: with 5000 rows that gives 1000 test mushrooms, enough for a stable estimate, while keeping as many rows as possible for training (the assignment asks us to use every row optimally). Hyperparameter tuning should use cross-validation **inside** the training set.
- `stratify` keeps the 62/38 class ratio in both sets.
- The test set must stay untouched until the final evaluation – also for experiments like removing suspected flipped labels (EDA section 9).

In [10]:
train_idx, test_idx = train_test_split(df.index, test_size=TEST_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE)
df["split"] = "train"
df.loc[test_idx, "split"] = "test"

pd.crosstab(df["split"], df[TARGET], normalize="index").round(3).assign(rows=df["split"].value_counts())

class,edible,poisonous,rows
split,,,
test,0.621,0.379,1000
train,0.622,0.378,4000


**Duplicates.** The EDA found no duplicate rows, but that was *with* the noise columns (whose shuffled values made every row unique) and *before* the stem rule filled in zeros. We check again now, per split, because identical rows in train and test would give a model a few test answers "for free".

In [11]:
features = NUM_COLS + ["has_stem"] + CAT_COLS
dup_rows = df[df.duplicated(subset=features, keep=False)]
n_groups = dup_rows.groupby(features, dropna=False).ngroups
print(f"Rows that have an identical twin (on all features): {len(dup_rows)} in {n_groups} groups")
print("Groups with conflicting labels:", (dup_rows.groupby(features, dropna=False)[TARGET].nunique() > 1).sum())
print("Groups that span train and test:", (dup_rows.groupby(features, dropna=False)["split"].nunique() > 1).sum())
dup_rows.sort_values(features)[features + [TARGET, "split"]]

Rows that have an identical twin (on all features): 16 in 7 groups
Groups with conflicting labels: 0
Groups that span train and test: 3


,cap_diameter,stem_height,stem_width,has_stem,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class,split
273,4.91,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,train
621,4.91,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,train
1969,NaN,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,test
2473,NaN,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,train
580,NaN,0.0,0.0,0,missing,missing,woods,autumn,none,others,none,poisonous,train
4082,NaN,0.0,0.0,0,missing,missing,woods,autumn,none,others,none,poisonous,test
232,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train
3164,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train
3720,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train
4244,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train


**Decision: keep the duplicates.** They are not copied records. 12 of the 16 rows are stemless, poisonous mushrooms of what looks like one species (`cap_shape = others`, woods, no stem); the other 4 are edible mushrooms with no size measurements at all. In both cases so many cells were blanked that different mushrooms have become indistinguishable. The dataset has many mushrooms per species, so this is expected. None of them have conflicting labels. Dropping them would thin out exactly the small, high-risk stemless group, and only a handful of groups span train and test (a few rows out of 1000 test rows, so at most a few tenths of a percent on the test score). In real use, the API would also receive such "look-alike" input.

We save the cleaned dataset. Numerical `NaN` are written as empty cells; categoricals already use `"missing"`. The raw file is not touched.

This file is what PyCaret / AutoML gets: those tools do their own imputation and encoding, and we can pass them our train and test rows via the `split` column.

In [12]:
CLEANED_COLS = NUM_COLS + ["has_stem"] + CAT_COLS + [TARGET, "split"]
df[CLEANED_COLS].to_csv(CLEANED_PATH, index=False)
print(f"Saved {CLEANED_PATH}  ({len(df):,} rows, {len(CLEANED_COLS)} columns)")
df[CLEANED_COLS].head()

Saved Data\mushroom_cleaned.csv  (5,000 rows, 13 columns)


,cap_diameter,stem_height,stem_width,has_stem,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class,split
0,NaN,6.42,13.51,1,missing,missing,woods,summer,grooved,convex,missing,poisonous,test
1,6.99,7.68,18.76,1,missing,white,woods,autumn,none,flat,smooth,edible,train
2,5.78,7.67,8.21,1,missing,green,woods,summer,none,conical,missing,poisonous,train
3,6.34,8.34,11.06,1,missing,missing,woods,autumn,zone,convex,missing,poisonous,train
4,NaN,5.67,12.78,1,missing,brown,woods,missing,none,convex,missing,edible,test


## 7. Preprocessing pipeline: imputation, normalisation, encoding
*(EDA sections 5.3, 6.2 and 7.1 – steps 6, 8 and 9)*

Everything that *learns from the data* goes into one scikit-learn `ColumnTransformer`, fitted on the training rows only:

| Columns | Steps | Why |
|---|---|---|
| `cap_diameter`, `stem_height`, `stem_width` | median imputation → `log1p` → standardisation | Missing at random (EDA 5.2), median is robust to the skew. `log1p` = log(1+x): removes the strong right skew (skewness 4.2 → 0.1) and still works for the zeros of stemless mushrooms. Standardising (mean 0, std 1) puts all features on the same scale, needed for logistic regression, KNN, SVM and neural networks. |
| `has_stem` | most-frequent imputation | Only the ~50 rows without any stem information are imputed; almost all mushrooms have a stem. |
| 7 categoricals | one-hot encoding, categories with < 10 training rows grouped into one "infrequent" column | No order (7.3). Very rare categories are unreliable (7.1) and give almost-empty columns. |

**Why one pipeline for all models?** `log1p` and standardisation are *monotonic*: they don't change the order of the values. Tree models (random forest, gradient boosting) only look at the order, so for them these steps change nothing, while linear and distance-based models need them. So one prepared dataset serves every model.

**Paths not taken:**
- *Rare-category threshold 30* (as first suggested in the EDA): it would also merge small but almost pure categories such as `stem_surface = shiny` (n=27, 93% poisonous) and `habitat = paths` (n=22, 82% poisonous) into one mixed group, which destroys real signal. With 10 we only merge categories that are too small to trust (e.g. spore colours purple/green, n=3). Section 9 compares both.
- *KNN / iterative imputation* of `cap_diameter` using its correlation with `stem_width` (ρ = 0.85): a promising experiment, but it belongs in the model notebooks where we can measure whether it improves the validation score. Median is the simple, robust default.
- *Outlier clipping*: not needed – after `log1p` the extremes are no longer extreme.

`handle_unknown="infrequent_if_exist"` has one more advantage for deployment: if the API receives a category the model never saw, it is treated as "infrequent" instead of crashing.

In [13]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, NUM_COLS),
        ("stem", SimpleImputer(strategy="most_frequent"), ["has_stem"]),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False), CAT_COLS),
    ],
    verbose_feature_names_out=True,
)

We fit the preprocessor on the training rows only, then use it to transform both sets.
`has_stem` is converted to a plain float first, because scikit-learn's imputer expects `NaN` rather than pandas' `pd.NA`.
The target becomes `is_poisonous` (1 = poisonous): poisonous is the class we care about most (recall!), so we make it the positive class.

In [14]:
def to_model_input(frame):
    # Select the feature columns in the order the preprocessor expects
    X = frame[NUM_COLS + ["has_stem"] + CAT_COLS].copy()
    X["has_stem"] = X["has_stem"].astype("float")
    return X

train, test = df[df["split"] == "train"], df[df["split"] == "test"]
X_train, X_test = to_model_input(train), to_model_input(test)
y_train = (train[TARGET] == "poisonous").astype(int).rename("is_poisonous")
y_test = (test[TARGET] == "poisonous").astype(int).rename("is_poisonous")

preprocessor.fit(X_train)
feature_names = preprocessor.get_feature_names_out()

prepared_train = pd.DataFrame(preprocessor.transform(X_train), columns=feature_names, index=X_train.index).join(y_train)
prepared_test = pd.DataFrame(preprocessor.transform(X_test), columns=feature_names, index=X_test.index).join(y_test)
print(f"{len(feature_names)} features after encoding")

64 features after encoding

Which categories were grouped as infrequent? This makes the effect of `min_frequency=10` visible.

In [15]:
encoder = preprocessor.named_transformers_["cat"]
{col: list(cats) for col, cats in zip(CAT_COLS, encoder.infrequent_categories_) if cats is not None}

{'spore_print_color': ['green', 'purple'], 'habitat': ['urban']}

We save the prepared sets and the fitted preprocessor. The original index is kept as `row_id`, so every prepared row can be traced back to the raw file (useful for error analysis later).
The preprocessor is saved with `joblib` (the scikit-learn recommended format; it is only a few KB, far below GitHub's 100 MB limit).

In [16]:
MODEL_DIR.mkdir(exist_ok=True)
prepared_train.to_csv(PREPARED_TRAIN_PATH, index_label="row_id")
prepared_test.to_csv(PREPARED_TEST_PATH, index_label="row_id")
joblib.dump(preprocessor, PREPROCESSOR_PATH)

for path in [PREPARED_TRAIN_PATH, PREPARED_TEST_PATH, PREPROCESSOR_PATH]:
    print(f"Saved {path}  ({path.stat().st_size / 1024:.0f} KB)")
prepared_train.head()

Saved Data\mushroom_prepared_train.csv  (1217 KB)


Saved Data\mushroom_prepared_test.csv  (305 KB)
Saved models\mushroom_preprocessor.joblib  (7 KB)


,num__cap_diameter,num__stem_height,num__stem_width,stem__has_stem,cat__spore_print_color_black,cat__spore_print_color_brown,cat__spore_print_color_gray,cat__spore_print_color_missing,cat__spore_print_color_pink,cat__spore_print_color_white,...,cat__stem_surface_fibrous,cat__stem_surface_grooves,cat__stem_surface_missing,cat__stem_surface_none,cat__stem_surface_scaly,cat__stem_surface_shiny,cat__stem_surface_silky,cat__stem_surface_smooth,cat__stem_surface_sticky,is_poisonous
1,0.289418,0.540953,0.813401,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,-0.079224,0.538032,-0.223408,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.098934,0.726646,0.142769,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5,0.049458,1.346808,0.145019,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
6,0.049458,0.575744,-0.208740,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## 8. Sanity checks

Automatic checks that the prepared data is what we intended. If any of them fails, the notebook stops.
1. No missing values left, and every row is kept (4000 + 1000 = 5000).
2. Train and test have exactly the same columns.
3. The class ratio is the same in both sets (stratification worked).
4. The numerical features have mean ≈ 0 and std ≈ 1 **on the training set**. (On the test set they will be close but not exact – that is expected, since the scaler never saw the test rows.)
5. Reading the saved files back gives the same data, and the saved preprocessor gives the same output (so the API will transform input exactly as during training).

In [17]:
assert prepared_train.notna().all().all() and prepared_test.notna().all().all()
assert len(prepared_train) + len(prepared_test) == len(df_raw)
assert list(prepared_train.columns) == list(prepared_test.columns)
assert abs(y_train.mean() - y_test.mean()) < 0.01

num_features = [f"num__{c}" for c in NUM_COLS]
print("Numerical features after preprocessing:")
print(pd.concat({"train mean": prepared_train[num_features].mean(), "train std": prepared_train[num_features].std(),
                 "test mean": prepared_test[num_features].mean(), "test std": prepared_test[num_features].std()}, axis=1).round(3))

reloaded = pd.read_csv(PREPARED_TRAIN_PATH, index_col="row_id")
assert np.allclose(reloaded.values, prepared_train.values)
assert np.allclose(joblib.load(PREPROCESSOR_PATH).transform(X_test), prepared_test[feature_names].values)
print("\nAll sanity checks passed.")

Numerical features after preprocessing:
                   train mean  train std  test mean  test std
num__cap_diameter         0.0        1.0     -0.045     0.952
num__stem_height          0.0        1.0      0.016     0.914
num__stem_width           0.0        1.0     -0.042     1.021

All sanity checks passed.


## 9. Does the preparation actually help?

The assignment asks to check that our changes actually improve the data. We compare a few versions of the preparation with the same model and the same **5-fold cross-validation on the training set only** (the test set stays untouched).

- Model: a random forest with default-ish settings. It is not our final model; it's a fixed "measuring instrument". We use a non-linear model because the EDA showed that is what this data needs (a quick check with logistic regression showed only differences of ±0.5 percentage points: a linear model is too weak here to benefit much from any of these steps).
- Metrics: accuracy, **recall of the poisonous class** (the costly error, EDA section 3) and ROC-AUC.

Versions:
1. `noise kept`: our full preparation, but with the two `jumbled_noise` columns added back.
2. `no stem fix`: noise dropped, but without the stem rule (stemless mushrooms with a missing measure get the median, no `has_stem`).
3. `rare threshold 30`: our preparation with `min_frequency=30` instead of 10.
4. `final`: the preparation of this notebook.

In [18]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate


def cv_scores(preproc, X, y):
    model = Pipeline([("prep", preproc),
                      ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "recall", "roc_auc"])
    return {m: scores[f"test_{m}"].mean() for m in ["accuracy", "recall", "roc_auc"]}


def variant(num_cols, stem_cols, cat_cols, min_frequency=10):
    # Same structure as the final preprocessor, with configurable columns
    steps = [("num", clone(numeric_pipeline), num_cols),
             ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency, sparse_output=False), cat_cols)]
    if stem_cols:
        steps.append(("stem", SimpleImputer(strategy="most_frequent"), stem_cols))
    return ColumnTransformer(steps)


# 1. Noise columns back in (taken from the raw data, same rows as the training set; NaN -> "missing")
X_noise = X_train.join(df_raw.loc[X_train.index, NOISE_COLS].fillna("missing"))
# 2. Stem measures as they were before the stem rule (translated and noise-free, but not repaired)
X_no_fix = X_train.copy()
X_no_fix[["stem_height", "stem_width"]] = df_raw.loc[X_train.index, ["stem-height", "stem-width"]].values
X_no_fix["stem_surface"] = df_raw.loc[X_train.index, "stem-surface"].map(CODE_MAPS["stem_surface"]).fillna("missing")

results = {
    "noise kept":        cv_scores(variant(NUM_COLS, ["has_stem"], CAT_COLS + NOISE_COLS), X_noise, y_train),
    "no stem fix":       cv_scores(variant(NUM_COLS, [], CAT_COLS), X_no_fix, y_train),
    "rare threshold 30": cv_scores(variant(NUM_COLS, ["has_stem"], CAT_COLS, min_frequency=30), X_train, y_train),
    "final":             cv_scores(preprocessor, X_train, y_train),
}
pd.DataFrame(results).T.round(3)

,accuracy,recall,roc_auc
noise kept,0.770,0.469,0.814
no stem fix,0.782,0.513,0.825
rare threshold 30,0.781,0.511,0.826
final,0.784,0.521,0.828


**Interpretation** *(numbers from our run; they can move by a few tenths of a percent between library versions)*
- **Dropping the noise columns clearly helps**: with them the forest loses about 1.5 points of accuracy and about 5 points of poisonous recall. A random forest picks features at random for its splits, so useless columns regularly get chosen and the model fits noise. This confirms the EDA conclusion with a model instead of a statistical test.
- **The stem rule helps a little** (< 1 point on every metric). It only changes about 20 rows and adds one feature, so a small effect is what we'd expect. We keep it because it is logically correct (a stemless mushroom should not get an invented stem) and it concerns the high-risk group (92% poisonous), not because of the score.
- **Threshold 10 vs. 30** for rare categories: 30 is slightly worse on every metric, in line with the reasoning in section 7.
- Apart from the noise columns, the differences are smaller than the variation between CV folds. So we read this as "these steps don't hurt and probably help", not as a proven improvement. The real comparison happens in the model notebooks.
- Poisonous recall is only around 50% at the default 0.5 threshold. That is a modelling problem, not a data problem: in the model notebooks we should lower the decision threshold and/or use class weights, since missing a poisonous mushroom is the costly error.

## 10. Summary & open questions for the model notebooks

### What was done (EDA decision table → implementation)
| # | EDA step | Done here |
|---|---|---|
| 1 | snake_case column names | ✅ section 2 |
| 2 | Letter codes → readable labels | ✅ section 2 (with a check for unmapped codes) |
| 3 | Drop `jumbled_noise_0/1` | ✅ section 3, benefit confirmed in section 9 |
| 4 | Keep all rows | ✅ 5000 in, 5000 out |
| 5 | Categorical NaN → `"missing"` | ✅ section 4 |
| 6 | Numerical NaN → median, fitted on train only | ✅ section 7 |
| 7 | Keep outliers & zero stems, add `has_stem` | ✅ section 5, extended with `stem_surface = none` and logical filling of missing stem measures |
| 8 | Group rare categories | ✅ section 7, threshold 10 (not 30, see section 7) |
| 9 | `log1p` + scaling | ✅ section 7, applied for all models since it is harmless for trees |
| 10 | Stratified train/test split | ✅ section 6, stored in the `split` column so every model uses the same test set |

### How to use the output in the model notebooks
```python
# Our own scikit-learn models: fully prepared, numeric data
train = pd.read_csv("Data/mushroom_prepared_train.csv", index_col="row_id")
X_train, y_train = train.drop(columns="is_poisonous"), train["is_poisonous"]

# PyCaret / AutoML: the cleaned data, the tool does its own imputation/encoding
cleaned = pd.read_csv("Data/mushroom_cleaned.csv")
train_df = cleaned[cleaned["split"] == "train"].drop(columns="split")
test_df = cleaned[cleaned["split"] == "test"].drop(columns="split")
```
For the API: apply the same cleaning (sections 2–5) to the user input, then `joblib.load("models/mushroom_preprocessor.joblib").transform(...)`.

### Open questions (to test in the model notebooks, on the training set with cross-validation)
- Drop `spore_print_color` (95% missing) or keep it as *present / missing*?
- Median vs. KNN / iterative imputation for `cap_diameter` (41% missing, ρ = 0.85 with `stem_width`).
- Remove or down-weight suspected flipped labels (EDA section 9) – **training set only**, and never remove *poisonous* labels just because a model disagrees.
- Feature engineering: `stem_width / stem_height`, cap / stem ratio.
- Decision threshold and class weights to raise recall on poisonous mushrooms.